In [ ]:
# load env
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# create client
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
# Helper to store the previous conversation
def add_user_message(messages, text):
    user_messages = {"role" : "user", "content" : text}
    messages.append(user_messages)

def add_assistant_message(messages, text):
    assistant_messages = {"role" : "assistant", "content" : text}
    messages.append(assistant_messages)

def chat(messages, system=None, temperature=1.0, stop_squences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_squences
    }

    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# get current date and time tool implementation
from datetime import datetime


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

In [ ]:
# schema for the get_current_datetime tool
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}
)

In [ ]:
# example usage of the get_current_datetime tool
messages = []

messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)
messages.append({
    "role": "assistant",
    "content": response.content
})
messages

In [ ]:
response.content[1].input

In [ ]:
# ** mean mappig fields to schema
result = get_current_datetime(**response.content[1].input)

In [ ]:
messages.append({
    "role" : "user",
    "content":[
        {
            "type": "tool_result",
            "tool_use_id": response.content[1].id,
            "content": result,
            "is_error": False
        }
    ]
})
messages

In [ ]:
# final call with existing conversation and tool result
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)